ilk basda python ve sqli ise salaq

In [5]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import text

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

from db import get_engine

engine = get_engine()

print("Python və SQL Server hazırdır!")

Python və SQL Server hazırdır!


csv fayllarimizi python data frame formasina getirek

In [6]:
import sys
from pathlib import Path

import pandas as pd

# Layihənin əsas qovluğu
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Notebook daxilindən db.py faylına çıxış
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

from db import get_engine

# CSV fayllarının yerləşdiyi qovluq
DATA_DIR = PROJECT_ROOT / "data" / "bank_data"

customers = pd.read_csv(DATA_DIR / "customers.csv")
loans = pd.read_csv(DATA_DIR / "loans.csv")
transactions = pd.read_csv(DATA_DIR / "transactions.csv")

print("Customers:", customers.shape)
print("Loans:", loans.shape)
print("Transactions:", transactions.shape)

Customers: (5000, 13)
Loans: (4820, 10)
Transactions: (77396, 8)


fayllari sql servere yazaq

In [7]:
from sqlalchemy import Unicode

customers.to_sql("customers", engine, schema="bronze", if_exists="replace",
                 index=False, dtype={c: Unicode(255) for c in customers.select_dtypes("string").columns})

loans.to_sql("loans", engine, schema="bronze", if_exists="replace",
             index=False, dtype={c: Unicode(255) for c in loans.select_dtypes("string").columns})

transactions.to_sql("transactions", engine, schema="bronze", if_exists="replace",
                    index=False, dtype={c: Unicode(255) for c in transactions.select_dtypes("string").columns})

106

Kredit və müştəri məlumatlarını birləşdirək

In [5]:
with engine.connect() as conn:
    customers_df = pd.read_sql(
        text("SELECT * FROM bronze.customers"), conn
    )

    loans_df = pd.read_sql(
        text("SELECT * FROM bronze.loans"), conn
    )

bronze_loans_merged = loans_df.merge(
    customers_df,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("Əvvəl kredit sayı:", len(loans_df))
print("Birləşmədən sonra:", len(bronze_loans_merged))

display(bronze_loans_merged.head())

Əvvəl kredit sayı: 4820
Birləşmədən sonra: 4820


,loan_id,customer_id,application_date,product_type,amount,term_months,interest_rate,status,rejection_reason,defaulted,...,city,branch,tenure_years,credit_score,balance,salary,products_count,has_card,is_active,churned
0,L000001,C03267,1/1/2026,Biznes krediti,18174.08,34,11.34,T?sdiql?ndi,NaN,0.0,...,Ming?çevir,Ming?çevir M?rk?z,5.0,682.0,15988.68,2175.84,2.0,1.0,1.0,0.0
1,L000002,C00082,1/2/2026,Istehlak krediti,14942.94,15,13.74,R?dd edildi,Yüks?k borc yükü,NaN,...,L?nk?ran,L?nk?ran M?rk?z,4.0,624.0,30895.36,2077.34,1.0,0.0,1.0,1.0
2,L000003,C00715,1/2/2026,Istehlak krediti,24217.34,58,14.09,T?sdiql?ndi,NaN,0.0,...,Baki,S?bail,5.0,700.0,1473.43,1912.54,2.0,0.0,0.0,1.0
3,L000004,C02969,1/2/2026,Biznes krediti,14890.76,58,14.82,R?dd edildi,Asagi kredit skoru,NaN,...,G?nc?,G?nc? M?rk?z,12.0,544.0,0.00,1020.38,1.0,0.0,1.0,1.0
4,L000005,C00178,1/3/2026,Istehlak krediti,6382.58,21,13.19,T?sdiql?ndi,NaN,0.0,...,L?nk?ran,L?nk?ran M?rk?z,6.0,850.0,6541.33,2214.41,1.0,1.0,1.0,0.0


Bayram təqvimini kreditlərə birləşdirək

In [6]:
# Bayram təqvimini Bronze-dan oxuyuruq
with engine.connect() as conn:
    calendar_df = pd.read_sql(
        text("SELECT * FROM bronze.calendar_reference"),
        conn
    )

# Tarixləri birləşdirmə üçün eyni formata gətiririk
bronze_loans_merged["merge_date"] = pd.to_datetime(
    bronze_loans_merged["application_date"]
).dt.date

calendar_df["merge_date"] = pd.to_datetime(
    calendar_df["date"]
).dt.date

# Bayram məlumatlarını kreditlərə əlavə edirik
bronze_loans_merged = bronze_loans_merged.merge(
    calendar_df[
        ["merge_date", "is_holiday", "holiday_name",
         "is_weekend", "is_workday", "is_pre_holiday"]
    ],
    on="merge_date",
    how="left",
    validate="many_to_one"
)

print("Birləşmədən sonra kredit sayı:", len(bronze_loans_merged))

display(
    bronze_loans_merged[
        ["loan_id", "application_date", "is_holiday",
         "holiday_name", "is_weekend"]
    ].head()
)

Birləşmədən sonra kredit sayı: 4820


,loan_id,application_date,is_holiday,holiday_name,is_weekend
0,L000001,1/1/2026,1,Yeni il,0
1,L000002,1/2/2026,1,Yeni il,0
2,L000003,1/2/2026,1,Yeni il,0
3,L000004,1/2/2026,1,Yeni il,0
4,L000005,1/3/2026,0,NaN,1


In [7]:
bronze_loans_merged.head()

,loan_id,customer_id,application_date,product_type,amount,term_months,interest_rate,status,rejection_reason,defaulted,...,products_count,has_card,is_active,churned,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday
0,L000001,C03267,1/1/2026,Biznes krediti,18174.08,34,11.34,T?sdiql?ndi,NaN,0.0,...,2.0,1.0,1.0,0.0,2026-01-01,1,Yeni il,0,0,0
1,L000002,C00082,1/2/2026,Istehlak krediti,14942.94,15,13.74,R?dd edildi,Yüks?k borc yükü,NaN,...,1.0,0.0,1.0,1.0,2026-01-02,1,Yeni il,0,0,0
2,L000003,C00715,1/2/2026,Istehlak krediti,24217.34,58,14.09,T?sdiql?ndi,NaN,0.0,...,2.0,0.0,0.0,1.0,2026-01-02,1,Yeni il,0,0,0
3,L000004,C02969,1/2/2026,Biznes krediti,14890.76,58,14.82,R?dd edildi,Asagi kredit skoru,NaN,...,1.0,0.0,1.0,1.0,2026-01-02,1,Yeni il,0,0,0
4,L000005,C00178,1/3/2026,Istehlak krediti,6382.58,21,13.19,T?sdiql?ndi,NaN,0.0,...,1.0,1.0,1.0,0.0,2026-01-03,0,NaN,1,0,0


Hava məlumatlarını birləşdirək

In [8]:
# Birləşdirmədən əvvəl sətir sayı
before_count = len(bronze_loans_merged)

# Hava məlumatlarını tarix və şəhər üzrə birləşdiririk
bronze_loans_merged = bronze_loans_merged.merge(
    weather_df[
        [
            "merge_date",
            "city",
            "temperature_2m_mean",
            "precipitation_sum",
            "wind_speed_10m_max"
        ]
    ],
    on=["merge_date", "city"],
    how="left",
    validate="many_to_one"
)

# Nəticəni yoxlayırıq
print("Əvvəlki kredit sayı:", before_count)
print("Birləşmədən sonrakı sətir sayı:", len(bronze_loans_merged))

print(
    "Hava məlumatı qoşulan sətirlər:",
    bronze_loans_merged["temperature_2m_mean"].notna().sum()
)

print(
    "Hava məlumatı qoşulmayan sətirlər:",
    bronze_loans_merged["temperature_2m_mean"].isna().sum()
)

display(
    bronze_loans_merged[
        [
            "loan_id",
            "application_date",
            "city",
            "temperature_2m_mean",
            "precipitation_sum",
            "wind_speed_10m_max"
        ]
    ].head(10)
)

NameError: name 'weather_df' is not defined

Hava məlumatı qoşulmayan 5 setri gorek

In [ ]:
display(
    bronze_loans_merged[
        bronze_loans_merged["temperature_2m_mean"].isna()
    ][
        ["loan_id", "application_date", "city"]
    ]
)

,loan_id,application_date,city
1402,L001398,2/17/2026,NaN
1754,L001749,2/27/2026,NaN
2672,L002665,4/9/2026,NaN
3698,L003687,5/15/2026,NaN
3795,L003784,5/20/2026,NaN


Birləşdirilmiş cədvəli Bronze-a yazaq

In [ ]:
# Birləşdirilmiş kredit məlumatlarını Bronze-a yazırıq
bronze_loans_merged.to_sql(
    name="loans_enriched",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False
)

# SQL Server-də sətir sayını yoxlayırıq
with engine.connect() as conn:
    result = pd.read_sql(
        text("""
            SELECT COUNT(*) AS row_count
            FROM bronze.loans_enriched
        """),
        conn
    )

display(result)

,row_count
0,4820


Tranzaksiya məlumatlarını yoxlayaq

In [ ]:
with engine.connect() as conn:
    transactions_df = pd.read_sql(
        text("SELECT * FROM bronze.transactions"),
        conn
    )

print("Tranzaksiya sayı:", len(transactions_df))
print("Sütunlar:", transactions_df.columns.tolist())

display(transactions_df.head())

Tranzaksiya sayı: 77396
Sütunlar: ['transaction_id', 'customer_id', 'date', 'time', 'amount', 'currency', 'category', 'channel']


,transaction_id,customer_id,date,time,amount,currency,category,channel
0,T0066501,C02139,6/1/2026,18:44:22,170.58,AZN,Köçürm?,Mobil t?tbiq
1,T0066502,C04131,6/1/2026,18:44:35,17.91,EUR,Alis-veris,Mobil t?tbiq
2,T0066503,C02018,6/1/2026,18:45:03,72.43,AZN,Mobil öd?nis,Mobil t?tbiq
3,T0066504,C01843,6/1/2026,18:46:09,9.48,AZN,Alis-veris,POS
4,T0066505,C03905,6/1/2026,18:46:13,53.60,AZN,Nagdlasdirma,ATM


Transactions + Customers

In [ ]:
# Tranzaksiyaları müştəri məlumatları ilə birləşdiririk
bronze_transactions_merged = transactions_df.merge(
    customers_df,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("Əvvəlki tranzaksiya sayı:", len(transactions_df))
print("Birləşmədən sonrakı sətir sayı:", len(bronze_transactions_merged))

print(
    "Müştəri məlumatı tapılmayan tranzaksiyalar:",
    bronze_transactions_merged["city"].isna().sum()
)

display(bronze_transactions_merged.head())

Əvvəlki tranzaksiya sayı: 77396
Birləşmədən sonrakı sətir sayı: 77396
Müştəri məlumatı tapılmayan tranzaksiyalar: 0


,transaction_id,customer_id,date,time,amount,currency,category,channel,age,gender,city,branch,tenure_years,credit_score,balance,salary,products_count,has_card,is_active,churned
0,T0066501,C02139,6/1/2026,18:44:22,170.58,AZN,Köçürm?,Mobil t?tbiq,37,Qadin,Baki,Bin?q?di,1,811,0.00,1156.10,1,1,1,0
1,T0066502,C04131,6/1/2026,18:44:35,17.91,EUR,Alis-veris,Mobil t?tbiq,28,Qadin,Ming?çevir,Ming?çevir M?rk?z,2,675,8457.92,2047.88,1,1,1,0
2,T0066503,C02018,6/1/2026,18:45:03,72.43,AZN,Mobil öd?nis,Mobil t?tbiq,26,Kisi,Baki,N?rimanov,6,674,9999.27,2531.39,1,0,1,0
3,T0066504,C01843,6/1/2026,18:46:09,9.48,AZN,Alis-veris,POS,38,Kisi,Baki,Yasamal,7,577,1322.47,856.51,2,1,0,0
4,T0066505,C03905,6/1/2026,18:46:13,53.60,AZN,Nagdlasdirma,ATM,52,Kisi,G?nc?,G?nc? M?rk?z,5,794,0.00,1221.73,2,1,1,0


Transactions+ Calendar
Tranzaksiya ile Təqvim məlumatlarını birləşdirək
İndi tranzaksiyalara bayram, həftəsonu və iş günü məlumatlarını əlavə edəcəyik.

In [ ]:
# Tranzaksiya tarixini birləşdirməyə hazırlayırıq
bronze_transactions_merged["merge_date"] = pd.to_datetime(
    bronze_transactions_merged["date"]
).dt.date

# Təqvim məlumatlarını birləşdiririk
bronze_transactions_merged = bronze_transactions_merged.merge(
    calendar_df[
        [
            "merge_date",
            "is_holiday",
            "holiday_name",
            "is_weekend",
            "is_workday",
            "is_pre_holiday"
        ]
    ],
    on="merge_date",
    how="left",
    validate="many_to_one"
)

print("Birləşmədən sonrakı sətir sayı:", len(bronze_transactions_merged))

print(
    "Təqvim məlumatı qoşulmayan sətirlər:",
    bronze_transactions_merged["is_holiday"].isna().sum()
)

display(
    bronze_transactions_merged[
        [
            "transaction_id",
            "date",
            "city",
            "is_holiday",
            "holiday_name",
            "is_weekend",
            "is_workday"
        ]
    ].head()
)

Birləşmədən sonrakı sətir sayı: 77396
Təqvim məlumatı qoşulmayan sətirlər: 0


,transaction_id,date,city,is_holiday,holiday_name,is_weekend,is_workday
0,T0066501,6/1/2026,Baki,0,NaN,0,1
1,T0066502,6/1/2026,Ming?çevir,0,NaN,0,1
2,T0066503,6/1/2026,Baki,0,NaN,0,1
3,T0066504,6/1/2026,Baki,0,NaN,0,1
4,T0066505,6/1/2026,G?nc?,0,NaN,0,1


Valyuta cədvəlini yoxlayaq
Bizdə bronze.exchange_rates cədvəlində AZN-dən USD və EUR-a məzənnələr var. Tranzaksiyalara bunları tarix və valyuta üzrə qoşacağıq.Amma əvvəlcə sütunları və valyutaları yoxlayaq.

In [ ]:
with engine.connect() as conn:
    exchange_df = pd.read_sql(
        text("SELECT * FROM bronze.exchange_rates"),
        conn
    )

print("Məzənnə sətirlərinin sayı:", len(exchange_df))
print("Sütunlar:", exchange_df.columns.tolist())

print("\nMəzənnə cədvəlindəki valyutalar:")
print(exchange_df["quote"].unique())

print("\nTranzaksiyalardakı valyutalar:")
print(bronze_transactions_merged["currency"].unique())

display(exchange_df.head())

Məzənnə sətirlərinin sayı: 362
Sütunlar: ['date', 'base', 'quote', 'rate']

Məzənnə cədvəlindəki valyutalar:
<ArrowStringArray>
['EUR', 'USD']
Length: 2, dtype: str

Tranzaksiyalardakı valyutalar:
<ArrowStringArray>
['AZN', 'EUR', 'USD']
Length: 3, dtype: str


,date,base,quote,rate
0,2026-01-01,AZN,EUR,0.50084
1,2026-01-01,AZN,USD,0.58837
2,2026-01-02,AZN,EUR,0.50095
3,2026-01-02,AZN,USD,0.58830
4,2026-01-03,AZN,EUR,0.50108


İndi məzənnələri tranzaksiyalara tarix və valyuta üzrə birləşdirəcəyik.

In [ ]:
# Məzənnə tarixini uyğunlaşdırırıq
exchange_df["merge_date"] = pd.to_datetime(
    exchange_df["date"]
).dt.date

# Birləşdirmə üçün ayrıca cədvəl hazırlayırıq
rates_for_merge = exchange_df[
    ["merge_date", "quote", "base", "rate"]
].copy()

rates_for_merge = rates_for_merge.rename(
    columns={
        "quote": "rate_currency",
        "base": "rate_base"
    }
)

# Tranzaksiyalarla məzənnələri birləşdiririk
bronze_transactions_merged = bronze_transactions_merged.merge(
    rates_for_merge,
    left_on=["merge_date", "currency"],
    right_on=["merge_date", "rate_currency"],
    how="left",
    validate="many_to_one"
)

print("Birləşmədən sonra sətir sayı:", len(bronze_transactions_merged))

print(
    "EUR/USD üçün məzənnəsi tapılmayan:",
    bronze_transactions_merged.loc[
        bronze_transactions_merged["currency"].isin(["EUR", "USD"]),
        "rate"
    ].isna().sum()
)

display(
    bronze_transactions_merged[
        ["transaction_id", "date", "amount", "currency", "rate"]
    ].head(10)
)

Birləşmədən sonra sətir sayı: 77396
EUR/USD üçün məzənnəsi tapılmayan: 0


,transaction_id,date,amount,currency,rate
0,T0066501,6/1/2026,170.58,AZN,NaN
1,T0066502,6/1/2026,17.91,EUR,0.50518
2,T0066503,6/1/2026,72.43,AZN,NaN
3,T0066504,6/1/2026,9.48,AZN,NaN
4,T0066505,6/1/2026,53.60,AZN,NaN
5,T0066506,6/1/2026,80.73,AZN,NaN
6,T0066507,6/1/2026,70.42,AZN,NaN
7,T0066508,6/1/2026,57.93,AZN,NaN
8,T0066509,6/1/2026,67.65,AZN,NaN
9,T0066510,6/1/2026,3.88,AZN,NaN


Hava məlumatlarını tranzaksiyalara əlavə edək

In [ ]:
# Birləşdirmədən əvvəl sətir sayı
before_count = len(bronze_transactions_merged)

# Hava məlumatlarını əlavə edirik
bronze_transactions_merged = bronze_transactions_merged.merge(
    weather_df[
        [
            "merge_date",
            "city",
            "temperature_2m_mean",
            "precipitation_sum",
            "wind_speed_10m_max"
        ]
    ],
    on=["merge_date", "city"],
    how="left",
    validate="many_to_one"
)

print("Əvvəlki tranzaksiya sayı:", before_count)
print("Birləşmədən sonrakı sətir sayı:", len(bronze_transactions_merged))

print(
    "Hava məlumatı qoşulmayan tranzaksiyalar:",
    bronze_transactions_merged["temperature_2m_mean"].isna().sum()
)

display(
    bronze_transactions_merged[
        [
            "transaction_id",
            "date",
            "city",
            "temperature_2m_mean",
            "precipitation_sum",
            "wind_speed_10m_max"
        ]
    ].head()
)

Əvvəlki tranzaksiya sayı: 77396
Birləşmədən sonrakı sətir sayı: 77396
Hava məlumatı qoşulmayan tranzaksiyalar: 0


,transaction_id,date,city,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,T0066501,6/1/2026,Baki,19.7,0.0,19.4
1,T0066502,6/1/2026,Ming?çevir,20.9,0.0,28.7
2,T0066503,6/1/2026,Baki,19.7,0.0,19.4
3,T0066504,6/1/2026,Baki,19.7,0.0,19.4
4,T0066505,6/1/2026,G?nc?,19.0,0.0,14.8


Birləşdirilmiş tranzaksiyaları Bronze-a yazaq

In [ ]:
# Birləşdirilmiş tranzaksiyaları SQL Server-ə yazırıq
bronze_transactions_merged.to_sql(
    name="transactions_enriched",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False
)

# SQL Server-də sətir sayını yoxlayırıq
with engine.connect() as conn:
    result = pd.read_sql(
        text("""
            SELECT COUNT(*) AS row_count
            FROM bronze.transactions_enriched
        """),
        conn
    )

display(result)

,row_count
0,77396


Bronze cədvəllərini yoxlayaq

In [ ]:
with engine.connect() as conn:
    bronze_tables = pd.read_sql(
        text("""
            SELECT TABLE_NAME
            FROM INFORMATION_SCHEMA.TABLES
            WHERE TABLE_SCHEMA = 'bronze'
            ORDER BY TABLE_NAME
        """),
        conn
    )

display(bronze_tables)

,TABLE_NAME
0,calendar_reference
1,customers
2,exchange_rates
3,loans
4,loans_enriched
5,transactions
6,transactions_enriched
7,weather
